In [5]:
import pandas as pd
import numpy as np
import pyshark
import asyncio
import ssl

DATA_FOLDER = "data"
PCAP_FILE = f"{DATA_FOLDER}/mom_3.pcap"
TOP_URLS_FILE = f"{DATA_FOLDER}/top_urls.csv"

OUTPUT_FOLDER = f"output"
OUTPUT_FILE = f"{OUTPUT_FOLDER}/mom_3.csv"

In [6]:
top_urls = pd.read_csv(TOP_URLS_FILE)
https_urls = top_urls[top_urls["url"].str.startswith("https://")]
https_urls

,url
2,https://18comic.vip
3,https://7games.bet.br
4,https://8moviesda.net
5,https://9animetv.to
6,https://9moviesda.com
...,...
995,https://zbporn.tv
996,https://zeenews.india.com
997,https://zh.m.wikipedia.org
998,https://zonatmo.com


In [7]:
context = ssl.create_default_context()
i = 1


async def good_url(url):
    domain = url.removeprefix("https://")
    try:
        _, writer = await asyncio.wait_for(asyncio.open_connection(domain, 443), timeout=25)
        await writer.start_tls(
            context, server_hostname=domain, ssl_handshake_timeout=5, ssl_shutdown_timeout=5
        )

        global i
        if i % 50 == 0:
            print(f"{i} good URLs so far...")
        i += 1

        return True
    except Exception as e:
        if str(e):
            print(f"Error connecting {domain}: {e}")
        else:
            print(f"Error connecting {domain}: {repr(e)}")
        return False


good_urls_indices = await asyncio.gather(*[good_url(url) for url in https_urls["url"].tolist()])
good_urls = https_urls[good_urls_indices]

Error connecting cdn.qaliy.com: [Errno 104] Connection reset by peer
50 good URLs so far...
100 good URLs so far...
Error connecting cricbet99.club: [Errno -5] No address associated with hostname
Error connecting cdn.gd85.com: [Errno 104] Connection reset by peer
Error connecting googll.store: [Errno 104] Connection reset by peer
150 good URLs so far...
200 good URLs so far...
Error connecting m.nkryu17dc.com: [Errno -3] Temporary failure in name resolution
Error connecting hrms.indianrail.gov.in: [Errno 111] Connect call failed ('203.176.112.88', 443)
Error connecting bollyflix.tw: [Errno -2] Name or service not known
Error connecting mangabuff.ru: [Errno 104] Connection reset by peer
250 good URLs so far...
Error connecting gall.dcinside.com: SSL handshake is taking longer than 5 seconds: aborting the connection
Error connecting moviebox.ng: [Errno -2] Name or service not known
Error connecting moviesda.it.com: [Errno -5] No address associated with hostname
Error connecting m.dcinsid

In [8]:
data = []
good_streams = set()
skipped_packets = 0


def process_server_hello(packet):
    good_streams.add(packet.tcp.stream)


def process_client_hello(packet):
    if packet.tcp.stream not in good_streams:
        global skipped_packets
        skipped_packets += 1
        return
    if not hasattr(packet.tls, "handshake_extensions_server_name"):
        print(f"Skipping packet without SNI")
        return
    data.append(
        {
            "timestamp": packet.sniff_time,
            "domain": packet.tls.handshake_extensions_server_name,
        }
    )


# We only want TLS Client Hello packets, but we want to skip streams for which the TLS
# handshake never concluded
cap_server_hello = pyshark.FileCapture(PCAP_FILE, display_filter="tls.handshake.type == 2")
await cap_server_hello.packets_from_tshark(process_server_hello)

print(f"Found {len(good_streams)} streams with Server Hello packets.")

cap_client_hello = pyshark.FileCapture(PCAP_FILE, display_filter="tls.handshake.type == 1")
await cap_client_hello.packets_from_tshark(process_client_hello)

print(f"{len(data)} TLS streams found. Skipped {skipped_packets} streams that never completed the handshake.")

Found 840 streams with Server Hello packets.
925 TLS streams found. Skipped 939 streams that never completed the handshake.


In [ ]:
data = pd.DataFrame(data)
# Express timestamps as second offset since the first packet
initial_time = data["timestamp"].min()
data["timestamp"] = (data["timestamp"] - initial_time).dt.total_seconds()
data

In [6]:
mapping = {}

good_urls = good_urls.sample(frac=1).reset_index(drop=True)
index = 0
for domain in data["domain"].unique():
    mapping[domain] = good_urls["url"].iloc[index]
    index += 1
    if index >= len(good_urls):
        index = 0

data_mapped = data.copy()
data_mapped["url"] = data_mapped["domain"].map(mapping)
data_mapped = data_mapped.drop(columns=["domain"])
data_mapped

,timestamp,url
0,0.000000,https://yandex.com
1,3.478341,https://ma.afribaba.com
2,8.119483,https://www.espncricinfo.com
3,9.129903,https://www.joyclub.de
4,9.481431,https://www.joyclub.de
...,...,...
920,14276.638800,https://www.haberler.com
921,14276.639166,https://www.haberler.com
922,14276.645345,https://articulo.mercadolibre.com.ar
923,14277.118654,https://www.haberler.com


In [7]:
BUCKET_SIZE = 30 * 60  # 30 mins

data_grouped = data_mapped
data_grouped["group"] = (data_grouped["timestamp"] // BUCKET_SIZE).astype(int)
data_grouped.iloc[np.r_[0:5, -5:0]]

,timestamp,url,group
0,0.000000,https://yandex.com,0
1,3.478341,https://ma.afribaba.com,0
2,8.119483,https://www.espncricinfo.com,0
3,9.129903,https://www.joyclub.de,0
4,9.481431,https://www.joyclub.de,0
920,14276.638800,https://www.haberler.com,7
921,14276.639166,https://www.haberler.com,7
922,14276.645345,https://articulo.mercadolibre.com.ar,7
923,14277.118654,https://www.haberler.com,7
924,14277.422471,https://www.haberler.com,7


In [9]:
BUCKETS_TO_KEEP = 3

# we will keep the most populous buckets
group_sizes = data_grouped.groupby("group").size()
data_pruned = data_grouped[
    data_grouped["group"].isin(group_sizes.nlargest(BUCKETS_TO_KEEP).index.tolist())
]
print(f"Largest bucket has {data_pruned.groupby('group').size().max()} packets")
print(f"Smallest bucket has {data_pruned.groupby('group').size().min()} packets")
data_pruned

Largest bucket has 338 packets
Smallest bucket has 190 packets


,timestamp,url,group
0,0.000000,https://yandex.com,0
1,3.478341,https://ma.afribaba.com,0
2,8.119483,https://www.espncricinfo.com,0
3,9.129903,https://www.joyclub.de,0
4,9.481431,https://www.joyclub.de,0
...,...,...,...
778,8814.500210,https://www.tamildhool.net,4
779,8815.311921,https://www.flightradar24.com,4
780,8816.035217,https://xn--42c5ab1a9aq9hqb5dud.com,4
781,8816.568212,https://xn--42c5ab1a9aq9hqb5dud.com,4


In [10]:
data_pruned.rename(columns={"group": "trace"}).to_csv(OUTPUT_FILE, index=False)

In [12]:
def remap_trace_index(traces):
    mapping = {}
    for i, trace_id in enumerate(traces["trace"].unique()):
        mapping[trace_id] = i
    traces["trace"] = traces["trace"].map(mapping)

def combine_traces(trace_file_1, trace_file_2):
    trace_1 = pd.read_csv(trace_file_1)
    remap_trace_index(trace_1)
    trace_2 = pd.read_csv(trace_file_2)
    remap_trace_index(trace_2)
    offset = trace_1["trace"].max() + 1
    trace_2["trace"] += offset
    return pd.concat([trace_1, trace_2], ignore_index=True)

In [17]:
traces = pd.read_csv("output/combined.csv")
for trace in traces["trace"].unique():
    mask = traces["trace"] == trace
    traces.loc[mask, "timestamp"] -= traces.loc[mask, "timestamp"].min()

traces.to_csv("output/combined.csv", index=False)